In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [2]:
# ==========================================
# 1. LOAD PRE-SPLIT DATASETS
# ==========================================
print("Loading datasets...")
train_df = pd.read_csv('train_orders.csv')
val_df = pd.read_csv('validation_orders.csv')
test_df = pd.read_csv('test_orders.csv')

Loading datasets...


In [3]:
# ==========================================
# 2. AUDIT: MISSING VALUES & DUPLICATES
# ==========================================
print("\n--- DATA QUALITY AUDIT ---")
for name, df in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    missing_count = df.isnull().sum().sum()
    duplicate_rows = df.duplicated().sum()
    duplicate_ids = df['Order Id'].duplicated().sum()
    print(f"[{name} Set] Missing Values: {missing_count} | Duplicate Rows: {duplicate_rows} | Duplicate Order IDs: {duplicate_ids}")

# Explicitly drop duplicate Order IDs if any existed
train_df = train_df.drop_duplicates(subset=['Order Id'])
val_df = val_df.drop_duplicates(subset=['Order Id'])
test_df = test_df.drop_duplicates(subset=['Order Id'])


--- DATA QUALITY AUDIT ---
[Train Set] Missing Values: 0 | Duplicate Rows: 0 | Duplicate Order IDs: 0
[Validation Set] Missing Values: 0 | Duplicate Rows: 0 | Duplicate Order IDs: 0
[Test Set] Missing Values: 0 | Duplicate Rows: 0 | Duplicate Order IDs: 0


Here, we have loaded the split datasets as train_orders, test_orders and validation_orders.
**Explanation:**
* **Data Integrity Check:** Performs defensive validation across all three splits to verify that there are zero missing values, zero duplicate rows, and zero duplicate `Order Id` entries.
* **Deduplication Safeguard:** Applies explicit `drop_duplicates` on `Order Id` as an intentional guardrail to guarantee strictly unique orders before feature transformation.

In [4]:
# ==========================================
# 3. SEPARATE FEATURES & TARGET
# ==========================================
X_train = train_df.drop(columns=['Order Id', 'target'])
y_train = train_df['target']

X_val = val_df.drop(columns=['Order Id', 'target'])
y_val = val_df['target']

X_test = test_df.drop(columns=['Order Id', 'target'])
y_test = test_df['target']

In [5]:
# ==========================================
# 4. DEFINE PREPROCESSING PIPELINES
# ==========================================
num_cols = [
    'scheduled_shipping_days', 'total_quantity', 'gross_sales',
    'total_order_value', 'total_discount', 'total_profit',
    'item_rows', 'unique_products', 'unique_categories',
    'order_year', 'order_month', 'order_day_of_week', 'order_hour', 'is_weekend'
]

cat_cols = [
    'shipping_mode', 'payment_type', 'customer_segment',
    'market', 'order_region', 'order_country'
]

# Imputer handles potential missing values defensively
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

**Explanation:**
* **Feature Categorization:** Groups attributes into 14 numerical variables (order totals, pricing, item counts, and temporal indicators) and 6 categorical variables (shipping modes, customer segments, payment methods, and regions).
* **Leakage Prevention:** High-cardinality features like `order_state` are intentionally excluded from one-hot encoding to prevent feature space explosion.
* **Pipeline Configuration:** Builds Scikit-Learn `Pipeline` and `ColumnTransformer` workflows incorporating `SimpleImputer` (defensive handling for unseen missing values), `StandardScaler` (z-score normalization for numerical inputs), and `OneHotEncoder` (dummy encoding for categorical inputs with `handle_unknown='ignore'`).

In [6]:
# ==========================================
# 5. FIT ON TRAIN ONLY & TRANSFORM
# ==========================================
print("\n--- TRANSFORMING DATASETS ---")
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print(f"X_train shape: {X_train_processed.shape}")
print(f"X_val shape:   {X_val_processed.shape}")
print(f"X_test shape:  {X_test_processed.shape}")


--- TRANSFORMING DATASETS ---
X_train shape: (46026, 212)
X_val shape:   (9863, 212)
X_test shape:  (9863, 212)


**Explanation:**
* **Data Leakage Mitigation:** Fits the `ColumnTransformer` strictly on `X_train` to prevent validation/test set statistics from leaking into the training phase.
* **Independent Transformation:** Applies the fitted transformer separately to `X_train`, `X_val`, and `X_test`, outputting dense, normalized feature matrices ready for model training.

In [7]:
# ==========================================
# 6. SAVE ARTIFACTS
# ==========================================
np.save('X_train.npy', X_train_processed)
np.save('X_val.npy', X_val_processed)
np.save('X_test.npy', X_test_processed)

y_train.to_csv('y_train.csv', index=False)
y_val.to_csv('y_val.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

joblib.dump(preprocessor, 'preprocessor.joblib')

print("\nProcessing complete! All arrays and preprocessor object saved successfully.")


Processing complete! All arrays and preprocessor object saved successfully.


**Explanation:**
* **Serialized Array Export:** Saves processed feature matrices (`X_train.npy`, `X_val.npy`, `X_test.npy`) as binary NumPy arrays for memory-efficient loading during model development.
* **Target & Transformer Preservation:** Exports target series as CSV files (`y_train.csv`, `y_val.csv`, `y_test.csv`) and serializes the fitted `ColumnTransformer` object (`preprocessor.joblib`) for deployment and inference on unseen test data.